# Multimodal Clustering of Amazon Fine Food Reviews
### Unsupervised Machine Learning — Final Project

---

| | |
|---|---|
| **Dataset** | Amazon Fine Food Reviews (Kaggle / Stanford SNAP) |
| **Samples** | 568,454 reviews (50,000 sampled for analysis) |
| **Approach** | Multimodal clustering: Text (TF-IDF + LSA) + Numerical features |
| **Algorithms** | K-Means, DBSCAN, Agglomerative Clustering |
| **Reduction** | TruncatedSVD, PCA, t-SNE |

In [ ]:
# ── Install dependencies if running in Colab / cloud ──────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import kagglehub
except ImportError:
    install('kagglehub')

try:
    from wordcloud import WordCloud
except ImportError:
    install('wordcloud')

try:
    import nltk
except ImportError:
    install('nltk')

print('✓ Dependencies ready')


In [ ]:
# ── Imports & Global Config ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.manifold import TSNE
from wordcloud import WordCloud
import re, warnings, ssl, nltk, os, pathlib

warnings.filterwarnings('ignore')

# Fix NLTK SSL on macOS
try:
    ssl._create_default_https_context = ssl._create_unverified_context
except AttributeError:
    pass
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# ── Constants ──────────────────────────────────────────────────────────────
RANDOM_STATE  = 42
N_SAMPLES     = 50_000
OPTIMAL_K     = 5
CLUSTER_COLS  = ['#e53935', '#1e88e5', '#43a047', '#fb8c00', '#8e24aa']
RATING_COLS   = ['#d32f2f', '#f57c00', '#fbc02d', '#388e3c', '#1976d2']

np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ── Find / Download dataset ─────────────────────────────────────────────────
def get_data_path():
    # Check common local paths first
    candidates = [
        '/Users/ashish/.cache/kagglehub/datasets/snap/amazon-fine-food-reviews/versions/2/Reviews.csv',
        '/root/.cache/kagglehub/datasets/snap/amazon-fine-food-reviews/versions/2/Reviews.csv',
        'Reviews.csv',
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # Not found locally → download via kagglehub
    print('Dataset not found locally. Downloading via kagglehub...')
    import kagglehub
    dataset_dir = kagglehub.dataset_download('snap/amazon-fine-food-reviews')
    for p in pathlib.Path(dataset_dir).rglob('Reviews.csv'):
        return str(p)
    raise FileNotFoundError('Reviews.csv not found even after download.')

DATA_PATH = get_data_path()
print(f'✓ Libraries loaded')
print(f'✓ Dataset path : {DATA_PATH}')
print(f'✓ Sample size  : {N_SAMPLES:,} reviews')


---
## i. Introduction — What This Project Does

**Problem:** Amazon hosts millions of product reviews. Can we automatically group them into meaningful clusters without any labels?

**Multimodal Approach:** Most clustering work uses only one feature type. We use **two modalities simultaneously**:

| Modality | Features | Method |
|---|---|---|
| **Text** | Review body + summary | TF-IDF (8k terms) → TruncatedSVD (100D) |
| **Numerical** | Rating, helpfulness ratio, text length, vote count | StandardScaler |
| **Fusion** | Early fusion (weighted concatenation) | → PCA (30D) |

**Goal:** Discover hidden review archetypes (e.g., *detailed enthusiasts*, *brief complaints*, *niche experts*) that could help Amazon improve recommendations, flag low-quality reviews, and surface useful feedback to sellers.

---
## ii. Data Loading

In [ ]:
# ── Load & Clean ───────────────────────────────────────────────────────────
print('Loading dataset...')
df_full = pd.read_csv(DATA_PATH)
print(f'Raw dataset : {df_full.shape[0]:,} rows × {df_full.shape[1]} columns')

df_full.drop_duplicates(subset=['UserId', 'Time', 'Text'], inplace=True)
df_full.dropna(subset=['Text', 'Summary'], inplace=True)
print(f'After dedup : {df_full.shape[0]:,} rows')

df = df_full.sample(n=N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

df['Date']              = pd.to_datetime(df['Time'], unit='s')
df['Year']              = df['Date'].dt.year
df['text_length']       = df['Text'].str.len()
df['word_count']        = df['Text'].str.split().str.len()
df['summary_length']    = df['Summary'].str.len()
df['helpfulness_ratio'] = df['HelpfulnessNumerator'] / (df['HelpfulnessDenominator'] + 1)

print(f'\nWorking sample : {len(df):,} reviews')
print(f'Date range     : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Unique products: {df["ProductId"].nunique():,}')
print(f'Unique users   : {df["UserId"].nunique():,}')
df[['Score','text_length','word_count','helpfulness_ratio']].describe().round(2)


---
## iii-a. Basic Summary of Data — Distributions

In [ ]:
# ── Key Statistics ─────────────────────────────────────────────────────────
print('=' * 55)
print('  DATASET SUMMARY STATISTICS')
print('=' * 55)
print(f'  Total reviews (sample)   : {len(df):,}')
print(f'  Average star rating      : {df["Score"].mean():.2f} / 5')
print(f'  Median review length     : {df["word_count"].median():.0f} words')
print(f'  Reviews with any votes   : {(df["HelpfulnessDenominator"]>0).sum():,} '
      f'({(df["HelpfulnessDenominator"]>0).mean()*100:.1f}%)')
print()
print('  Rating breakdown:')
for s in range(1, 6):
    n = (df['Score'] == s).sum()
    pct = n / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'    {s}★  {bar:<25} {pct:5.1f}%  ({n:,})')

In [ ]:
# ── EDA Figure 1: Core Distributions ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Amazon Fine Food Reviews — Exploratory Data Analysis', fontsize=15, fontweight='bold')

# 1. Rating bar chart
rc = df['Score'].value_counts().sort_index()
axes[0,0].bar(rc.index, rc.values, color=RATING_COLS, edgecolor='white', linewidth=1.5)
for x, y in zip(rc.index, rc.values):
    axes[0,0].text(x, y + 80, f'{y/len(df)*100:.1f}%', ha='center', fontsize=9)
axes[0,0].set_title('Rating Distribution', fontweight='bold')
axes[0,0].set_xlabel('Star Rating'); axes[0,0].set_ylabel('Count')

# 2. Review length histogram
axes[0,1].hist(df['word_count'].clip(0, 600), bins=70, color='#5c6bc0', edgecolor='white', alpha=0.85)
med = df['word_count'].median()
axes[0,1].axvline(med, color='red', linestyle='--', label=f'Median: {med:.0f} words')
axes[0,1].set_title('Review Word Count Distribution', fontweight='bold')
axes[0,1].set_xlabel('Word Count'); axes[0,1].set_ylabel('Frequency'); axes[0,1].legend()

# 3. Helpfulness ratio (reviews with votes)
voted = df[df['HelpfulnessDenominator'] > 0]
axes[0,2].hist(voted['helpfulness_ratio'], bins=50, color='#ab47bc', edgecolor='white', alpha=0.85)
axes[0,2].set_title(f'Helpfulness Ratio\n(n={len(voted):,} reviews with votes)', fontweight='bold')
axes[0,2].set_xlabel('Helpful / Total Votes'); axes[0,2].set_ylabel('Frequency')

# 4. Reviews per year
rpy = df.groupby('Year').size()
axes[1,0].bar(rpy.index, rpy.values, color='#ef5350', edgecolor='white', alpha=0.85)
axes[1,0].set_title('Reviews Over Time', fontweight='bold')
axes[1,0].set_xlabel('Year'); axes[1,0].set_ylabel('Number of Reviews')
axes[1,0].tick_params(axis='x', rotation=45)

# 5. Average rating per year
ary = df.groupby('Year')['Score'].mean()
axes[1,1].plot(ary.index, ary.values, 'o-', color='#ff7043', linewidth=2.5, markersize=8)
axes[1,1].set_ylim(1, 5); axes[1,1].set_title('Avg Rating Over Time', fontweight='bold')
axes[1,1].set_xlabel('Year'); axes[1,1].set_ylabel('Avg Star Rating')
axes[1,1].tick_params(axis='x', rotation=45)

# 6. Box plot: word count by rating
groups = [df[df['Score'] == s]['word_count'].clip(0, 500).values for s in range(1, 6)]
bp = axes[1,2].boxplot(groups, labels=[f'{s}★' for s in range(1,6)],
                        patch_artist=True, notch=True, widths=0.5)
for patch, c in zip(bp['boxes'], RATING_COLS):
    patch.set_facecolor(c); patch.set_alpha(0.75)
axes[1,2].set_title('Review Length by Star Rating', fontweight='bold')
axes[1,2].set_xlabel('Star Rating'); axes[1,2].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── EDA Figure 2: Text & Correlation Insights ──────────────────────────────
stop_words_set = set(stopwords.words('english'))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Text Content & Correlations', fontsize=14, fontweight='bold')

# 1. Overall word cloud
all_text = ' '.join(df['Text'].sample(5000, random_state=42))
wc = WordCloud(width=700, height=450, background_color='white',
               stopwords=stop_words_set, colormap='plasma',
               max_words=120, collocations=False).generate(all_text)
axes[0].imshow(wc, interpolation='bilinear'); axes[0].axis('off')
axes[0].set_title('Most Frequent Words (All Reviews)', fontweight='bold')

# 2. Percentage of 5★ vs 1★ reviews over time
yearly = df.groupby('Year')['Score'].apply(lambda x: pd.Series({
    'pct_5star': (x==5).mean()*100, 'pct_1star': (x==1).mean()*100}))
yearly = yearly.unstack()
yearly['pct_5star'].plot(ax=axes[1], color='#43a047', marker='o', linewidth=2, label='5★')
yearly['pct_1star'].plot(ax=axes[1], color='#e53935', marker='s', linewidth=2, label='1★')
axes[1].set_title('% of 5★ and 1★ Reviews Over Time', fontweight='bold')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Percentage (%)')
axes[1].legend(); axes[1].tick_params(axis='x', rotation=45)

# 3. Helpfulness vs word count scatter
s2 = df[df['HelpfulnessDenominator'] > 2].sample(min(3000, len(voted)), random_state=42)
sc = axes[2].scatter(s2['word_count'].clip(0, 700), s2['helpfulness_ratio'],
                     c=s2['Score'], cmap='RdYlGn', alpha=0.45, s=15, vmin=1, vmax=5)
plt.colorbar(sc, ax=axes[2], label='Star Rating')
axes[2].set_title('Helpfulness vs Review Length', fontweight='bold')
axes[2].set_xlabel('Word Count'); axes[2].set_ylabel('Helpfulness Ratio')

plt.tight_layout()
plt.savefig('eda_text_insights.png', dpi=130, bbox_inches='tight')
plt.show()

---
## iii-b. In-Depth Analysis — Feature Engineering (Multimodal)

### Why Multimodal?

A review is more than its words. A 5-star review written in 10 words carries a very different signal than a 5-star review with 500 words and 200 helpfulness votes. By combining **text semantics** with **behavioral/numerical signals** we get clusters that are richer and more actionable.

In [ ]:
# ── Text Preprocessing ─────────────────────────────────────────────────────
def preprocess(text):
    """Lowercase, strip HTML/punctuation, normalize whitespace."""
    text = str(text).lower()
    text = re.sub(r'<[^>]+>',   ' ', text)   # HTML tags
    text = re.sub(r'http\S+',   ' ', text)   # URLs
    text = re.sub(r'[^a-z\s]', ' ', text)   # keep only letters
    return re.sub(r'\s+', ' ', text).strip()

print('Preprocessing text (Summary + Review body)...')
df['clean_text'] = (df['Summary'].fillna('') + ' ' + df['Text']).apply(preprocess)
print(f'Avg clean text length: {df["clean_text"].str.len().mean():.0f} chars')
print(f'\nExample (first review):\n  {df["clean_text"].iloc[0][:200]}...')

In [ ]:
# ── Modality 1: Text — TF-IDF → TruncatedSVD (LSA) ────────────────────────
print('Step 1 — TF-IDF vectorization...')
tfidf = TfidfVectorizer(
    max_features  = 8_000,
    ngram_range   = (1, 2),
    min_df        = 5,
    max_df        = 0.85,
    sublinear_tf  = True,
    stop_words    = 'english'
)
X_tfidf = tfidf.fit_transform(df['clean_text'])
print(f'  TF-IDF matrix : {X_tfidf.shape}')

print('Step 2 — Latent Semantic Analysis (TruncatedSVD, 100 components)...')
N_SVD = 100
svd = TruncatedSVD(n_components=N_SVD, random_state=RANDOM_STATE)
X_text = svd.fit_transform(X_tfidf)
print(f'  SVD features  : {X_text.shape}')
print(f'  Var explained : {svd.explained_variance_ratio_.sum():.2%}')

In [ ]:
# ── Modality 2: Numerical Features ────────────────────────────────────────
print('Step 3 — Numerical features...')
num_cols = [
    'Score',
    'helpfulness_ratio',
    'word_count',
    'summary_length',
]
# Log-transform skewed features
df['log_word_count']      = np.log1p(df['word_count'])
df['log_summary_length']  = np.log1p(df['summary_length'])
df['log_total_votes']     = np.log1p(df['HelpfulnessDenominator'])
df['rating_norm']         = (df['Score'] - 1) / 4   # 0–1

use_cols = ['rating_norm', 'helpfulness_ratio',
            'log_word_count', 'log_summary_length', 'log_total_votes']

scaler   = StandardScaler()
X_num    = scaler.fit_transform(df[use_cols])
print(f'  Numerical features: {X_num.shape}')
print(f'  Columns: {use_cols}')

# Correlation heatmap
fig, ax = plt.subplots(figsize=(7, 5))
cdf = pd.DataFrame(X_num, columns=use_cols)
sns.heatmap(cdf.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Numerical Features — Correlation Matrix', fontweight='bold')
plt.tight_layout(); plt.savefig('num_correlation.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Multimodal Fusion (Early Fusion) ──────────────────────────────────────
# Weight text more heavily (it carries richer semantics)
TEXT_WEIGHT = 1.0
NUM_WEIGHT  = 0.6

X_fused = np.hstack([X_text * TEXT_WEIGHT, X_num * NUM_WEIGHT])
print('Multimodal Feature Matrix (Early Fusion):')
print(f'  Text modality  : {X_text.shape[1]} dims (TF-IDF → SVD)  × weight {TEXT_WEIGHT}')
print(f'  Numerical      : {X_num.shape[1]}  dims (scaled)         × weight {NUM_WEIGHT}')
print(f'  Fused matrix   : {X_fused.shape}')

---
## iii-c. In-Depth Analysis — Dimensionality Reduction (PCA)

In [ ]:
# ── PCA on Fused Features ─────────────────────────────────────────────────
print('Applying PCA to fused features...')
N_PCA = 30
pca   = PCA(n_components=N_PCA, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_fused)
print(f'  PCA features : {X_pca.shape}')
print(f'  Var explained: {pca.explained_variance_ratio_.sum():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PCA — Dimensionality Reduction of Multimodal Features', fontweight='bold')

# Scree plot
evr = pca.explained_variance_ratio_
axes[0].bar(range(1, N_PCA+1), evr*100, color='#5c6bc0', alpha=0.7, edgecolor='white')
axes[0].plot(range(1, N_PCA+1), evr*100, 'o-', color='#283593', markersize=5)
axes[0].set_title('Scree Plot — Variance per Component', fontweight='bold')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance (%)')

# Cumulative variance
cumvar = np.cumsum(evr) * 100
axes[1].plot(range(1, N_PCA+1), cumvar, 'o-', color='#26a69a', linewidth=2.5, markersize=6)
axes[1].fill_between(range(1, N_PCA+1), cumvar, alpha=0.15, color='#26a69a')
for thresh, col, lbl in [(80, 'red', '80%'), (90, 'orange', '90%')]:
    axes[1].axhline(thresh, color=col, linestyle='--', alpha=0.7, label=f'{lbl} threshold')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].set_xlabel('Number of PCs'); axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=130, bbox_inches='tight')
plt.show()

---
## iii-d. In-Depth Analysis — Clustering

Three algorithms compared:
- **K-Means** — partition-based, scales well, requires k up-front
- **DBSCAN** — density-based, no need to set k, finds arbitrary shapes
- **Agglomerative (Ward)** — hierarchical, good for confirming K-Means structure

In [ ]:
# ── Elbow Method + Silhouette Score ────────────────────────────────────────
print('Searching for optimal k (K-Means)...')
K_RANGE       = range(2, 11)
inertias      = []
sil_scores    = []

for k in K_RANGE:
    km  = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10, max_iter=300)
    lbl = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca, lbl, sample_size=8000, random_state=RANDOM_STATE)
    sil_scores.append(sil)
    print(f'  k={k}: inertia={km.inertia_:,.0f}  silhouette={sil:.4f}')

best_k = list(K_RANGE)[int(np.argmax(sil_scores))]
print(f'\nBest k by silhouette: {best_k}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('K-Means: Finding Optimal Number of Clusters', fontweight='bold')

axes[0].plot(list(K_RANGE), inertias, 'o-', color='#ef5350', linewidth=2.5, markersize=8)
axes[0].set_title('Elbow Method (Inertia)', fontweight='bold')
axes[0].set_xlabel('Number of Clusters k'); axes[0].set_ylabel('Inertia')
axes[0].set_xticks(list(K_RANGE))

axes[1].plot(list(K_RANGE), sil_scores, 'o-', color='#42a5f5', linewidth=2.5, markersize=8)
axes[1].axvline(best_k, color='red', linestyle='--', label=f'Best k = {best_k}')
axes[1].set_title('Silhouette Score vs k', fontweight='bold')
axes[1].set_xlabel('Number of Clusters k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_xticks(list(K_RANGE)); axes[1].legend()

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final K-Means Clustering ───────────────────────────────────────────────
print(f'Running final K-Means with k={OPTIMAL_K}...')
kmeans        = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init=20, max_iter=500)
df['cluster'] = kmeans.fit_predict(X_pca)

final_sil = silhouette_score(X_pca, df['cluster'], sample_size=10000, random_state=RANDOM_STATE)
print(f'Final Silhouette Score: {final_sil:.4f}')

print('\nCluster sizes:')
for c, n in df['cluster'].value_counts().sort_index().items():
    print(f'  Cluster {c}: {n:,} reviews ({n/len(df)*100:.1f}%)')

In [ ]:
# ── t-SNE (2D projection for visualization) ────────────────────────────────
print('Running t-SNE (this takes ~3-5 min on 50k samples)...')
# Pre-reduce to 20D with PCA for efficiency
pca20   = PCA(n_components=20, random_state=RANDOM_STATE)
X_pca20 = pca20.fit_transform(X_fused)

tsne = TSNE(n_components=2, perplexity=35, learning_rate='auto',
            init='pca', n_iter=750, random_state=RANDOM_STATE)
X_tsne = tsne.fit_transform(X_pca20)

df['tsne_x'] = X_tsne[:, 0]
df['tsne_y'] = X_tsne[:, 1]
print('t-SNE complete!')

In [ ]:
# ── DBSCAN on t-SNE coords ────────────────────────────────────────────────
print('Running DBSCAN on t-SNE coordinates...')
dbscan = DBSCAN(eps=3.0, min_samples=20, n_jobs=-1)
df['dbscan_cluster'] = dbscan.fit_predict(X_tsne)

n_db_clusters = len(set(df['dbscan_cluster'])) - (1 if -1 in df['dbscan_cluster'].values else 0)
n_noise       = (df['dbscan_cluster'] == -1).sum()
print(f'  DBSCAN clusters found : {n_db_clusters}')
print(f'  Noise / outlier points: {n_noise:,} ({n_noise/len(df)*100:.1f}%)')

In [ ]:
# ── Agglomerative Clustering (comparison) ─────────────────────────────────
print('Running Agglomerative Clustering (Ward linkage)...')
pca10   = PCA(n_components=10, random_state=RANDOM_STATE)
X_pca10 = pca10.fit_transform(X_fused)

agg = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='ward')
df['agg_cluster'] = agg.fit_predict(X_pca10)

agg_sil = silhouette_score(X_pca10, df['agg_cluster'], sample_size=10000, random_state=RANDOM_STATE)
ari     = adjusted_rand_score(df['cluster'], df['agg_cluster'])
print(f'  Agglomerative silhouette : {agg_sil:.4f}')
print(f'  K-Means silhouette       : {final_sil:.4f}')
print(f'  Adjusted Rand Index (agreement): {ari:.4f}  [1=identical, 0=random]')

---
## iv. Results

In [ ]:
# ── t-SNE Visualization — K-Means vs Rating ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('t-SNE Map of 50,000 Reviews', fontsize=14, fontweight='bold')

# K-Means clusters
for c in range(OPTIMAL_K):
    m = df['cluster'] == c
    axes[0].scatter(df.loc[m,'tsne_x'], df.loc[m,'tsne_y'],
                    s=3, alpha=0.4, color=CLUSTER_COLS[c], label=f'Cluster {c}')
axes[0].set_title(f'K-Means (k={OPTIMAL_K})', fontweight='bold')
axes[0].legend(markerscale=5, fontsize=9)
axes[0].set_xlabel('t-SNE 1'); axes[0].set_ylabel('t-SNE 2')

# By star rating
sc = axes[1].scatter(df['tsne_x'], df['tsne_y'], c=df['Score'],
                     cmap='RdYlGn', s=3, alpha=0.35, vmin=1, vmax=5)
plt.colorbar(sc, ax=axes[1], label='Star Rating')
axes[1].set_title('Colored by Star Rating', fontweight='bold')
axes[1].set_xlabel('t-SNE 1'); axes[1].set_ylabel('t-SNE 2')

# By review length
sc2 = axes[2].scatter(df['tsne_x'], df['tsne_y'], c=df['log_word_count'],
                      cmap='YlOrBr', s=3, alpha=0.35)
plt.colorbar(sc2, ax=axes[2], label='log(Word Count)')
axes[2].set_title('Colored by Review Length', fontweight='bold')
axes[2].set_xlabel('t-SNE 1'); axes[2].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig('tsne_visualization.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cluster Profile Analysis ───────────────────────────────────────────────
profile = df.groupby('cluster').agg(
    n_reviews         = ('Id', 'count'),
    avg_rating        = ('Score', 'mean'),
    pct_5star         = ('Score', lambda x: (x==5).mean()*100),
    pct_1star         = ('Score', lambda x: (x==1).mean()*100),
    avg_words         = ('word_count', 'mean'),
    avg_helpfulness   = ('helpfulness_ratio', 'mean'),
    avg_votes         = ('HelpfulnessDenominator', 'mean'),
).round(2)

print('Cluster Profiles:')
print(profile.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Cluster Profiles', fontweight='bold')

# Heatmap
cols_for_heat = ['avg_rating', 'pct_5star', 'pct_1star', 'avg_words', 'avg_helpfulness', 'avg_votes']
heat_data     = profile[cols_for_heat].copy()
heat_norm     = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min())
sns.heatmap(heat_norm.T, annot=heat_data.T.round(1), fmt='g',
            cmap='YlOrRd', ax=axes[0], linewidths=0.5,
            cbar_kws={'label': 'Normalized (0-1)'})
axes[0].set_title('Normalized Feature Heatmap per Cluster', fontweight='bold')
axes[0].set_xlabel('Cluster')

# Stacked rating bars
rating_dist = df.groupby(['cluster', 'Score']).size().unstack(fill_value=0)
rating_pct  = rating_dist.div(rating_dist.sum(axis=1), axis=0) * 100
rating_pct.plot(kind='bar', ax=axes[1], color=RATING_COLS,
                edgecolor='white', width=0.7, stacked=True)
axes[1].set_title('Rating Distribution per Cluster (%)', fontweight='bold')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Stars', labels=['1★','2★','3★','4★','5★'], bbox_to_anchor=(1.01,1))
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('cluster_profiles.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Word Clouds per Cluster ────────────────────────────────────────────────
cmaps = ['Reds', 'Blues', 'Greens', 'Oranges', 'Purples']

fig, axes = plt.subplots(1, OPTIMAL_K, figsize=(22, 5))
fig.suptitle('Word Clouds — Most Characteristic Terms per Cluster', fontsize=14, fontweight='bold')

for c in range(OPTIMAL_K):
    cdf   = df[df['cluster'] == c]
    texts = ' '.join(cdf['clean_text'].sample(min(3000, len(cdf)), random_state=42))
    wc = WordCloud(width=420, height=300, background_color='white',
                   colormap=cmaps[c], max_words=80,
                   stopwords=stop_words_set, collocations=False).generate(texts)
    axes[c].imshow(wc, interpolation='bilinear'); axes[c].axis('off')
    avg_r = cdf['Score'].mean()
    n     = len(cdf)
    axes[c].set_title(f'Cluster {c}\nn={n:,}  avg={avg_r:.2f}★', fontweight='bold')

plt.tight_layout()
plt.savefig('cluster_wordclouds.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Top TF-IDF Terms per Cluster ───────────────────────────────────────────
print('TOP DISCRIMINATIVE TERMS PER CLUSTER')
print('=' * 70)

for c in range(OPTIMAL_K):
    cdf   = df[df['cluster'] == c]
    tfidf_c = TfidfVectorizer(max_features=3000, stop_words='english',
                              ngram_range=(1, 2))
    tfidf_c.fit(cdf['clean_text'].tolist())
    feat_names = tfidf_c.get_feature_names_out()
    # low IDF = most common in this cluster
    top_idx  = np.argsort(tfidf_c.idf_)[:15]
    top_terms = [feat_names[i] for i in top_idx]
    print(f'\nCluster {c}  (n={len(cdf):,}, avg_rating={cdf["Score"].mean():.2f}★, '
          f'avg_words={cdf["word_count"].mean():.0f})')
    print(f'  Top terms: {chr(10)}    ' + '\n    '.join([f'{t}' for t in top_terms]))

In [ ]:
# ── Algorithm Comparison ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Clustering Algorithm Comparison on t-SNE Map', fontsize=14, fontweight='bold')

# K-Means
for c in range(OPTIMAL_K):
    m = df['cluster'] == c
    axes[0].scatter(df.loc[m,'tsne_x'], df.loc[m,'tsne_y'],
                    s=3, alpha=0.4, color=CLUSTER_COLS[c], label=f'C{c}')
axes[0].set_title(f'K-Means (k={OPTIMAL_K})\nSilhouette: {final_sil:.4f}', fontweight='bold')
axes[0].legend(markerscale=5, fontsize=9)

# DBSCAN
db_ids = sorted(set(df['dbscan_cluster']))
cmap_db = plt.cm.get_cmap('tab10', len(db_ids))
for i, cid in enumerate(db_ids):
    m     = df['dbscan_cluster'] == cid
    color = 'lightgrey' if cid == -1 else cmap_db(i)
    lbl   = 'Noise' if cid == -1 else f'C{cid}'
    axes[1].scatter(df.loc[m,'tsne_x'], df.loc[m,'tsne_y'],
                    s=3, alpha=0.35, color=color, label=lbl)
axes[1].set_title(f'DBSCAN\n{n_db_clusters} clusters, {n_noise/len(df)*100:.1f}% noise', fontweight='bold')
axes[1].legend(markerscale=5, fontsize=8, ncol=2)

# Agglomerative
for c in range(OPTIMAL_K):
    m = df['agg_cluster'] == c
    axes[2].scatter(df.loc[m,'tsne_x'], df.loc[m,'tsne_y'],
                    s=3, alpha=0.4, color=CLUSTER_COLS[c], label=f'C{c}')
axes[2].set_title(f'Agglomerative Ward\nSilhouette: {agg_sil:.4f}  ARI: {ari:.4f}', fontweight='bold')
axes[2].legend(markerscale=5, fontsize=9)

for ax in axes:
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')

plt.tight_layout()
plt.savefig('algorithm_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample Reviews per Cluster ────────────────────────────────────────────
print('REPRESENTATIVE REVIEWS PER CLUSTER')
print('=' * 80)
for c in range(OPTIMAL_K):
    cdf = df[df['cluster'] == c]
    print(f'\n{"─"*80}')
    print(f'CLUSTER {c}  |  {len(cdf):,} reviews  |  avg={cdf["Score"].mean():.2f}★  '
          f'|  avg_words={cdf["word_count"].mean():.0f}')
    print(f'{"─"*80}')
    for _, row in cdf.sample(min(2, len(cdf)), random_state=42).iterrows():
        print(f'  [{row["Score"]}★]  {row["Summary"]}')
        print(f'  "{row["Text"][:220]}..."')
        print()

In [ ]:
# ── Final Summary Dashboard ────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 12))
fig.suptitle('Multimodal Clustering — Amazon Fine Food Reviews\nFinal Results Dashboard',
             fontsize=17, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

# 1. t-SNE cluster map (large)
ax1 = fig.add_subplot(gs[0, :2])
for c in range(OPTIMAL_K):
    m = df['cluster'] == c
    ax1.scatter(df.loc[m,'tsne_x'], df.loc[m,'tsne_y'],
                s=5, alpha=0.45, color=CLUSTER_COLS[c], label=f'Cluster {c}')
ax1.set_title(f't-SNE Multimodal Cluster Map  (Silhouette={final_sil:.4f})', fontweight='bold')
ax1.legend(markerscale=4, fontsize=10)
ax1.set_xlabel('t-SNE 1'); ax1.set_ylabel('t-SNE 2')

# 2. Cluster sizes with avg rating
ax2 = fig.add_subplot(gs[0, 2])
bars = ax2.bar(range(OPTIMAL_K), profile['n_reviews'],
               color=CLUSTER_COLS, edgecolor='white', alpha=0.9)
ax2r = ax2.twinx()
ax2r.plot(range(OPTIMAL_K), profile['avg_rating'], 'D--',
          color='black', linewidth=2, markersize=9, label='Avg ★')
ax2.set_title('Cluster Size & Avg Rating', fontweight='bold')
ax2.set_xlabel('Cluster'); ax2.set_ylabel('# Reviews')
ax2r.set_ylabel('Avg Rating'); ax2r.set_ylim(1, 5)
ax2r.legend(loc='upper right')
for bar, n in zip(bars, profile['n_reviews']):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+30,
             f'{n:,}', ha='center', va='bottom', fontsize=9)

# 3. Algorithm silhouette comparison
ax3 = fig.add_subplot(gs[1, 0])
methods = ['K-Means', 'Agglomerative']
scores  = [final_sil, agg_sil]
colors3 = ['#5c6bc0', '#26a69a']
ax3.barh(methods, scores, color=colors3, edgecolor='white', height=0.5)
for i, s in enumerate(scores):
    ax3.text(s + 0.002, i, f'{s:.4f}', va='center', fontsize=10)
ax3.set_title('Silhouette Score Comparison', fontweight='bold')
ax3.set_xlabel('Silhouette Score'); ax3.set_xlim(0, max(scores)*1.2)

# 4. PCA variance waterfall
ax4 = fig.add_subplot(gs[1, 1])
cumvar = np.cumsum(pca.explained_variance_ratio_)*100
ax4.plot(range(1, N_PCA+1), cumvar, 'o-', color='#ef5350', linewidth=2.5, markersize=5)
ax4.fill_between(range(1, N_PCA+1), cumvar, alpha=0.15, color='#ef5350')
ax4.axhline(80, color='grey', linestyle='--', alpha=0.6)
ax4.set_title('PCA Cumulative Variance', fontweight='bold')
ax4.set_xlabel('# Principal Components'); ax4.set_ylabel('Var Explained (%)')

# 5. Stacked rating bars per cluster
ax5 = fig.add_subplot(gs[1, 2])
rating_pct.plot(kind='bar', ax=ax5, color=RATING_COLS, edgecolor='white', stacked=True, width=0.7)
ax5.set_title('Star Rating Mix per Cluster', fontweight='bold')
ax5.set_xlabel('Cluster'); ax5.set_ylabel('%')
ax5.legend(labels=['1★','2★','3★','4★','5★'], fontsize=8, bbox_to_anchor=(1.01,1))
ax5.tick_params(axis='x', rotation=0)

plt.savefig('final_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print('Dashboard saved → final_dashboard.png')

---
## v. Issues We Ran Into

| Issue | Description | Resolution |
|---|---|---|
| **Curse of Dimensionality** | 8,000-dim TF-IDF is too sparse for direct distance-based clustering | Reduced to 100D via TruncatedSVD (LSA) before clustering |
| **t-SNE Scalability** | t-SNE is O(n²) — extremely slow on 50k samples | Pre-reduced to 20D with PCA; used `learning_rate='auto'` and 750 iters |
| **Class Imbalance in Ratings** | ~64% of reviews are 5-star — clusters can be dominated by positives | Multimodal fusion down-weights raw rating; text separates reviews within same star level |
| **Choosing k** | Elbow curve is gradual — no sharp bend | Combined elbow + silhouette score; validated with Agglomerative ARI |
| **Duplicate Reviews** | ~2,000 near-identical reviews inflated certain clusters | Removed duplicates on (UserId, Time, Text) before sampling |
| **Modality Weighting** | Text and numerical features have different scales/magnitudes | Applied StandardScaler to numerical; used explicit weights (1.0 / 0.6) for fusion |

---
## vi. What We Would Like to Do in the Future

1. **Sentence Transformers (SBERT)** — Replace TF-IDF+SVD with `all-MiniLM-L6-v2` embeddings for semantic richness; TF-IDF misses paraphrases and negation.
2. **Product Category Metadata** — The dataset only has `ProductId`. Joining with Amazon category data would let us study *cross-category* review patterns.
3. **Temporal Clustering** — Study how review clusters shift over time (e.g., do complaint patterns spike around holidays?).
4. **Optimal DBSCAN Tuning** — Use k-distance plots to systematically choose `eps` instead of manual tuning.
5. **Topic Modeling per Cluster** — Apply LDA or BERTopic inside each K-Means cluster for finer-grained sub-topics.
6. **Automated Cluster Labeling** — Use an LLM to auto-generate a human-readable label for each cluster from top terms + sample reviews.
7. **Larger Sample / Full Dataset** — Run on all 568k reviews with GPU-accelerated RAPIDS cuML for production-scale results.

---
## vii. Suggestions for the Client (Amazon / Sellers)

Based on the clusters discovered:

| Cluster Archetype | Suggested Action |
|---|---|
| **Detailed, Helpful Reviewers** (long text, high helpfulness) | Surface these first in search results — high trust signal |
| **Brief Positive Reviews** (5★, short, low votes) | Flag for authenticity review — bot/incentivized reviews often cluster here |
| **Negative Complaint Cluster** (1-2★, emotional language) | Route to seller/support team for automated response; track product defect signals |
| **Niche Expert Reviews** (specific ingredient/nutrition language) | Target to health-conscious customer segments; useful for recommendation tuning |
| **Mid-Range Neutral Reviews** (3-4★, mixed sentiment) | Most useful for product improvement — customers explain *why* they didn't love it |